In [10]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage
from rich import print as rprint


# 导入大语言模型
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking": {"type": "disabled"}
    }
)

# 定义状态
class OverAllState(MessagesState):
    username: str
    output: str

# 定义结点
def node_1(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是" + state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

# 构件图
builder = StateGraph(state_schema = OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "llm_node")
builder.add_edge("llm_node", END)

# 编译
graph = builder.compile()

# 运行
result = graph.invoke({"username": "江来"})
rprint(result)

{
    'messages': [
        HumanMessage(
            content='你好，我是江来',
            additional_kwargs={},
            response_metadata={},
            id='f5b5b333-bf7c-469d-b27e-d8046720adb8'
        ),
        AIMessage(
            content='你好，江来！很高兴认识你。😊\n\n有什么我可以帮你的吗？无论是聊天、解答问题、还是需要一些建议，
我都很乐意帮忙！',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 34,
                    'prompt_tokens': 9,
                    'total_tokens': 43,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 9
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '80f22487-bdc2-4891-a397-551fa094318f',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a16419-4614-7434-b120-2ffba5efbacd-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 9,
                'output_tokens': 34,
                'total_tokens': 43,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    'username': '江来',
    'output': 
'你好，江来！很高兴认识你。😊\n\n有什么我可以帮你的吗？无论是聊天、解答问题、还是需要一些建议，我都很乐意帮忙！'
}